In [ ]:
import os
from Bio import Phylo


def rename_all_leaves(tree):
    leaf_index = 1
    for clade in tree.find_clades(terminal=True):
        clade.name = f"{leaf_index}leaf"
        print(f"Renamed leaf node to '{clade.name}'")
        leaf_index += 1


def process_nwk_file(file_path):
    print(f"Processing file: {file_path}")
    tree = Phylo.read(file_path, "newick")
    rename_all_leaves(tree)
    Phylo.write(tree, file_path, "newick")
    print(f"Updated file: {file_path}")


def traverse_and_process(root_dir):
    for subdir, _, files in os.walk(root_dir):
        for file in files:
            if file.endswith(".nwk"):
                file_path = os.path.join(subdir, file)
                process_nwk_file(file_path)


if __name__ == "__main__":
    root_directory = "/home/ganesank/project/phytclust/benchmarking_results/PhytClust_input/Case2_K10_N100_simulation"  # Change this to your directory
    traverse_and_process(root_directory)

In [ ]:
import re

# Define file paths
input_file = (
    "/home/ganesank/project/PhyCLIP/supplement_material_msz053/Figure_S3.tre.txt"
)
output_file = (
    "/home/ganesank/project/PhyCLIP/supplement_material_msz053/Figure_S3.newick"
)


# Function to parse the Translate block and sanitize labels
def parse_translate_block(nexus_content):
    translate_block = re.search(
        r"Begin trees;\s*Translate\s*(.*?)\s*;", nexus_content, re.DOTALL
    ).group(1)
    translate_lines = translate_block.split(",")
    translate_dict = {}
    for line in translate_lines:
        match = re.match(r"\s*(\d+)\s+\'?([^\'\s]+)\'?\s*", line.strip())
        if match:
            sanitized_name = (
                match.group(2).replace("/", "_").replace("*", "").replace(" ", "_")
            )
            translate_dict[match.group(1)] = sanitized_name
    return translate_dict


# Function to replace numeric identifiers with sanitized taxon names in the Newick tree
def replace_identifiers_in_tree(nexus_content, translate_dict):
    tree_block = re.search(r"tree\s+\w+\s*=\s*(.*?);", nexus_content, re.DOTALL).group(
        1
    )
    for num, name in translate_dict.items():
        tree_block = re.sub(r"\b" + re.escape(num) + r"\b", name, tree_block)
    return tree_block


# Read the Nexus file
with open(input_file, "r") as file:
    nexus_content = file.read()

# Parse and sanitize the Translate block
translate_dict = parse_translate_block(nexus_content)

# Replace numeric identifiers in the Newick tree with sanitized names
newick_tree = replace_identifiers_in_tree(nexus_content, translate_dict)

# Write the Newick tree to the output file
with open(output_file, "w") as file:
    file.write(newick_tree)

print(f"Converted {input_file} to {output_file} in Newick format.")

In [ ]:
import os
import pandas as pd


def rename_chrom_column_in_files(files):
    for file_path in files:
        try:
            df = pd.read_csv(file_path, sep="\t")
            if "chrom" in df.columns:
                df.rename(columns={"chrom": "chr"}, inplace=True)
                df.to_csv(file_path, sep="\t", index=False)
                print(f"Renamed column in {os.path.basename(file_path)}")
            else:
                print(f"No 'chrom' column in {os.path.basename(file_path)}")
        except Exception as e:
            print(f"Failed to process {os.path.basename(file_path)}: {e}")


# List of specific files to process
files_to_process = [
    "/home/ganesank/project/phytclust/Data/Roland_2015/processed_chunks/OV03-03_sorted.tsv",
    # Add other missed files here
]

# Call the function
rename_chrom_column_in_files(files_to_process)

In [ ]:
import os
import pandas as pd


def rename_chrom_column_in_files(directory):
    for filename in os.listdir(directory):
        if filename.endswith("_sorted.tsv"):
            file_path = os.path.join(directory, filename)
            df = pd.read_csv(file_path, sep="\t")
            if "chrom" in df.columns:
                df.rename(columns={"chrom": "chr"}, inplace=True)
                df.to_csv(file_path, sep="\t", index=False)
                print(f"Renamed column in {filename}")


# Specify the directory containing the files
directory = "/home/ganesank/project/phytclust/Data/Roland_2015/processed_chunks"

# Call the function
rename_chrom_column_in_files(directory)

In [ ]:
def label_internal_nodes_with_confidence(tree):
    for node in tree.find_clades():
        if not node.is_terminal():
            # Set the label of the internal node to its confidence value
            if node.confidence is not None:
                node.name = str(node.confidence)
            else:
                node.name = ""


label_internal_nodes_with_confidence(tree)